# Mortgage Extraction & Property Database Merge

Extracts mortgage instrument data (MORTGAGE / DISCHARGE OF MORTGAGE entries)
from Alberta Land Title Certificate PDFs, then left-joins the results onto an
existing property database export — producing a single ready-to-use Excel
file with no manual XLOOKUP required.

**How to use this notebook**
1. Run the cells from top to bottom.
2. Edit the variables in the **Configuration** cell to point at your Drive
   folder / files before each run.
3. Part 1 (PDF parsing) and Part 2 (database merge) are kept in separate,
   independently-rerunnable sections — see the section headers below. If you
   only need to fix a merge issue, you can rerun Part 2 without
   re-parsing every PDF (as long as `mortgage_results` is still in memory).


## 0. Setup

In [ ]:
# Install dependencies (pdfplumber is preferred; PyPDF2 is a fallback).
!pip install -q pdfplumber openpyxl


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Configuration

Edit these paths for each run. `PDF_FOLDER_PATH` should point at the Drive
folder containing the title PDFs (named `{PID}_filename.pdf`).
`PROPERTY_DB_PATH` is the existing property database export (must contain a
`Prop ID` column). `OUTPUT_PATH` is where the merged result will be written
— it will NOT overwrite the original database file.


In [ ]:
# ====================== CONFIGURATION — EDIT PER RUN ======================

# Folder in Google Drive containing the title PDFs, named "{PID}_filename.pdf"
PDF_FOLDER_PATH = "/content/drive/MyDrive/Gettel/Title Documents"

# Existing property database export (.xlsx) with a "Prop ID" column
PROPERTY_DB_PATH = "/content/drive/MyDrive/Gettel/property_database.xlsx"

# Where to write the merged output (a NEW file — the original is never overwritten)
OUTPUT_PATH = "/content/drive/MyDrive/Gettel/property_database_with_mortgages.xlsx"

# Name of the join key column in the property database
PROP_ID_COLUMN = "Prop ID"

# ============================================================================


In [ ]:
import os
import re
import glob
import difflib
import traceback
from collections import defaultdict
from datetime import datetime

import pandas as pd

try:
    import pdfplumber
    PDF_BACKEND = "pdfplumber"
except ImportError:
    import PyPDF2
    PDF_BACKEND = "pypdf2"

print(f"Using PDF backend: {PDF_BACKEND}")


## 2. Part 1 — Extract mortgage data from PDFs

This section is self-contained: it reads PDFs from `PDF_FOLDER_PATH` and
produces `mortgage_results` (a dict of `PID -> list of mortgage dicts`) plus
logs of anything that needs manual review. Rerun just this section if you
change the parsing logic.


### 2.1 PDF text extraction

In [ ]:
def extract_pdf_text(pdf_path):
    """Extract full text from a PDF using pdfplumber, falling back to PyPDF2."""
    if PDF_BACKEND == "pdfplumber":
        text_parts = []
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text_parts.append(page.extract_text() or "")
        return "\n".join(text_parts)
    else:
        text_parts = []
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                text_parts.append(page.extract_text() or "")
        return "\n".join(text_parts)


### 2.2 Locate and split the ENCUMBRANCES, LIENS & INTERESTS section

In [ ]:
ENCUMBRANCE_HEADER_RE = re.compile(r"ENCUMBRANCES,?\s*LIENS\s*&\s*INTERESTS", re.IGNORECASE)

# A new entry starts with: registration number, registration date, entry type
# e.g. "182 058 105   09/03/2018   MORTGAGE" or "...  DISCHARGE OF MORTGAGE"
ENTRY_START_RE = re.compile(
    r"^(?P<regnum>\d{3}\s*\d{3}\s*\d{3})\s+"
    r"(?P<date>\d{2}/\d{2}/\d{4})\s+"
    r"(?P<etype>[A-Z][A-Z /]*[A-Z])\s*$"
)


def get_encumbrance_section(full_text):
    """Return the document text starting at the ENCUMBRANCES section header."""
    match = ENCUMBRANCE_HEADER_RE.search(full_text)
    if not match:
        return ""
    return full_text[match.end():]


def split_into_entries(section_text):
    """
    Split the encumbrance section into entries.
    Each entry: {regnum, date, etype, lines: [particulars lines]}
    """
    entries = []
    current = None

    for raw_line in section_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        m = ENTRY_START_RE.match(line)
        if m:
            if current:
                entries.append(current)
            current = {
                "regnum": re.sub(r"\s+", " ", m.group("regnum")).strip(),
                "date": m.group("date"),
                "etype": m.group("etype").strip(),
                "lines": [],
            }
        elif current is not None:
            current["lines"].append(line)

    if current:
        entries.append(current)
    return entries


### 2.3 Parse MORTGAGE and DISCHARGE OF MORTGAGE entries

In [ ]:
LENDER_RE = re.compile(r"MORTGAGEE\s*-\s*(.+)", re.IGNORECASE)
AMOUNT_RE = re.compile(r"ORIGINAL\s+PRINCIPAL\s+AMOUNT\s*:?\s*\$?([\d,]+(?:\.\d+)?)", re.IGNORECASE)
DISCHARGE_REF_RE = re.compile(
    r"\bRE\b\s*:?\s*[A-Za-z]*\s*(\d{3}\s*\d{3}\s*\d{3})", re.IGNORECASE
)
REGNUM_ANYWHERE_RE = re.compile(r"\d{3}\s*\d{3}\s*\d{3}")


def normalize_regnum(s):
    return re.sub(r"\s+", "", s)


def parse_date(d):
    try:
        return datetime.strptime(d, "%d/%m/%Y")
    except Exception:
        return None


def parse_mortgage_entry(entry):
    """Extract lender / amount from a MORTGAGE entry's particulars block."""
    lender, amount = None, None
    for line in entry["lines"]:
        if lender is None:
            lm = LENDER_RE.search(line)
            if lm:
                lender = lm.group(1).strip().rstrip(".")
        if amount is None:
            am = AMOUNT_RE.search(line)
            if am:
                amount = float(am.group(1).replace(",", ""))
    return {
        "regnum": entry["regnum"],
        "regnum_norm": normalize_regnum(entry["regnum"]),
        "date": entry["date"],
        "lender": lender,
        "amount": amount,
        "discharged": "No",  # default; updated by discharge matching below
    }


def parse_discharge_entry(entry):
    """
    Extract whatever the discharge entry gives us to identify the mortgage it
    discharges: a referenced instrument number (preferred) and/or a mortgagee
    name. Format is unconfirmed, so both are best-effort. Matched per line
    (not on the joined block) so a greedy lender match can't swallow
    unrelated address/footer lines that follow it.
    """
    ref_regnum = None
    ref_lender = None
    for line in entry["lines"]:
        if ref_regnum is None:
            rm = DISCHARGE_REF_RE.search(line)
            if rm:
                ref_regnum = normalize_regnum(rm.group(1))
        if ref_lender is None:
            lm = LENDER_RE.search(line)
            if lm:
                ref_lender = lm.group(1).strip().rstrip(".")

    if ref_regnum is None:
        for line in entry["lines"]:
            rm2 = REGNUM_ANYWHERE_RE.search(line)
            if rm2:
                ref_regnum = normalize_regnum(rm2.group(0))
                break

    return {
        "regnum": entry["regnum"],
        "date": entry["date"],
        "ref_regnum": ref_regnum,
        "ref_lender": ref_lender,
    }


### 2.4 Match discharges to mortgages (instrument ref → fuzzy name+date → Unknown)

In [ ]:
def normalize_name(name):
    if not name:
        return ""
    name = name.upper()
    name = re.sub(r"[^A-Z0-9 ]", "", name)
    return re.sub(r"\s+", " ", name).strip()


def names_match(a, b, threshold=0.8):
    na, nb = normalize_name(a), normalize_name(b)
    if not na or not nb:
        return False
    if na == nb or na in nb or nb in na:
        return True
    return difflib.SequenceMatcher(None, na, nb).ratio() >= threshold


def match_discharges_to_mortgages(mortgages, discharges):
    """
    Mutates `mortgages` in place: sets 'discharged' to 'Yes' on confident
    matches. For discharges that can't be confidently tied to one mortgage,
    any not-yet-discharged mortgage that could plausibly be the target is
    marked 'Unknown' (never silently left as 'No'), and the discharge entry
    is returned for manual-review logging.
    """
    unmatched = []

    for disc in discharges:
        matched = None
        disc_date = parse_date(disc["date"])

        # Method 1: direct instrument-number reference
        if disc["ref_regnum"]:
            for mtg in mortgages:
                if mtg["regnum_norm"] == disc["ref_regnum"]:
                    matched = mtg
                    break

        # Method 2: fuzzy mortgagee name + chronological order
        if matched is None and disc["ref_lender"]:
            candidates = []
            for mtg in mortgages:
                if not names_match(mtg["lender"], disc["ref_lender"]):
                    continue
                mtg_date = parse_date(mtg["date"])
                if disc_date and mtg_date and mtg_date > disc_date:
                    continue  # a discharge can't predate its mortgage
                candidates.append(mtg)
            if candidates:
                candidates.sort(key=lambda m: parse_date(m["date"]) or datetime.min, reverse=True)
                matched = candidates[0]

        if matched is not None:
            matched["discharged"] = "Yes"
        else:
            # Could not confidently identify which mortgage this discharges.
            # Don't guess "No" -- flag plausible candidates as "Unknown".
            for mtg in mortgages:
                if mtg["discharged"] == "Yes":
                    continue
                mtg_date = parse_date(mtg["date"])
                if disc_date and mtg_date and mtg_date > disc_date:
                    continue
                mtg["discharged"] = "Unknown"
            unmatched.append(disc)

    return unmatched


### 2.5 Per-PDF driver + batch processing (errors isolated per file)

In [ ]:
def process_pdf(pdf_path):
    """
    Parse one PDF. Returns (pid, mortgages, unmatched_discharges).
    Raises on unrecoverable errors -- caller is responsible for catching.
    """
    filename = os.path.basename(pdf_path)
    pid = filename.split("_")[0]

    full_text = extract_pdf_text(pdf_path)
    section_text = get_encumbrance_section(full_text)
    entries = split_into_entries(section_text)

    mortgages, discharges = [], []
    for entry in entries:
        etype = entry["etype"].upper()
        if etype == "MORTGAGE":
            mortgages.append(parse_mortgage_entry(entry))
        elif "DISCHARGE" in etype and "MORTGAGE" in etype:
            discharges.append(parse_discharge_entry(entry))
        # all other entry types (CAVEAT, TRANSFER, etc.) are ignored

    unmatched = match_discharges_to_mortgages(mortgages, discharges)
    return pid, mortgages, unmatched


def run_extraction(pdf_folder):
    """
    Process every PDF in pdf_folder. One bad file never kills the batch.
    Returns: mortgage_results, failed_pdfs, unmatched_discharge_log, mortgage_count_buckets
    """
    pdf_paths = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))

    mortgage_results = {}
    failed_pdfs = []
    unmatched_discharge_log = []
    mortgage_count_buckets = defaultdict(int)

    for pdf_path in pdf_paths:
        filename = os.path.basename(pdf_path)
        try:
            pid, mortgages, unmatched = process_pdf(pdf_path)
            mortgage_results[pid] = mortgages

            n = len(mortgages)
            mortgage_count_buckets[n if n < 2 else "2+"] += 1

            for disc in unmatched:
                unmatched_discharge_log.append((filename, pid, disc))

        except Exception as e:
            failed_pdfs.append((filename, f"{type(e).__name__}: {e}"))
            traceback.print_exc()

    return mortgage_results, failed_pdfs, unmatched_discharge_log, mortgage_count_buckets


In [ ]:
mortgage_results, failed_pdfs, unmatched_discharge_log, mortgage_count_buckets = run_extraction(
    PDF_FOLDER_PATH
)
print(f"Succeeded: {len(mortgage_results)} PDFs | Failed: {len(failed_pdfs)} PDFs")


### 2.6 Pivot into a wide table (one row per PID, dynamic Mortgage_N columns)

In [ ]:
def build_wide_table(mortgage_results):
    max_mortgages = max((len(v) for v in mortgage_results.values()), default=0)

    rows = []
    for pid, mortgages in mortgage_results.items():
        row = {"PID": pid}
        ordered = sorted(mortgages, key=lambda m: parse_date(m["date"]) or datetime.min)
        for i in range(max_mortgages):
            n = i + 1
            if i < len(ordered):
                m = ordered[i]
                row[f"Mortgage{n}_Lender"] = m["lender"]
                row[f"Mortgage{n}_Date"] = m["date"]
                row[f"Mortgage{n}_Amount"] = m["amount"]
                row[f"Mortgage{n}_RegNum"] = m["regnum"]
                row[f"Mortgage{n}_Discharged"] = m["discharged"]
            else:
                row[f"Mortgage{n}_Lender"] = None
                row[f"Mortgage{n}_Date"] = None
                row[f"Mortgage{n}_Amount"] = None
                row[f"Mortgage{n}_RegNum"] = None
                row[f"Mortgage{n}_Discharged"] = None
        rows.append(row)

    return pd.DataFrame(rows), max_mortgages


mortgage_wide_df, max_mortgages_found = build_wide_table(mortgage_results)
print(f"Max mortgages found on a single title: {max_mortgages_found}")
mortgage_wide_df.head()


## 3. Part 2 — Merge into the property database

Self-contained: only needs `mortgage_wide_df` from Part 1 (already in
memory) and `PROPERTY_DB_PATH`. Rerun just this section after fixing a
parsing edge case, without re-processing every PDF.


In [ ]:
def merge_with_database(db_path, mortgage_wide_df, prop_id_col=PROP_ID_COLUMN):
    db_df = pd.read_excel(db_path, dtype={prop_id_col: str})
    db_df[prop_id_col] = db_df[prop_id_col].astype(str).str.strip()

    mtg_df = mortgage_wide_df.copy()
    mtg_df["PID"] = mtg_df["PID"].astype(str).str.strip()

    # Left join: every database row is preserved, row order is preserved,
    # and the new Mortgage_N columns are appended to the right.
    merged = db_df.merge(mtg_df, left_on=prop_id_col, right_on="PID", how="left")
    merged = merged.drop(columns=["PID"])  # redundant with prop_id_col

    db_ids = set(db_df[prop_id_col])
    pdf_ids = set(mtg_df["PID"])

    db_ids_without_pdf = sorted(db_ids - pdf_ids)
    pdf_ids_without_db_match = sorted(pdf_ids - db_ids)

    return merged, db_ids_without_pdf, pdf_ids_without_db_match


merged_df, db_ids_without_pdf, pdf_ids_without_db_match = merge_with_database(
    PROPERTY_DB_PATH, mortgage_wide_df
)
merged_df.head()


In [ ]:
merged_df.to_excel(OUTPUT_PATH, index=False, engine="openpyxl")
print(f"Wrote merged file to: {OUTPUT_PATH}")


## 4. Processing summary

In [ ]:
print("=" * 70)
print("PROCESSING SUMMARY")
print("=" * 70)

total_pdfs = len(mortgage_results) + len(failed_pdfs)
print(f"\nTotal PDFs processed: {total_pdfs}")
print(f"  Succeeded: {len(mortgage_results)}")
print(f"  Failed:    {len(failed_pdfs)}")

print("\nMortgage count breakdown:")
print(f"  0 mortgages:  {mortgage_count_buckets.get(0, 0)}")
print(f"  1 mortgage:   {mortgage_count_buckets.get(1, 0)}")
print(f"  2+ mortgages: {mortgage_count_buckets.get('2+', 0)}")

print(f"\nPDFs that failed to parse entirely ({len(failed_pdfs)}):")
if failed_pdfs:
    for fname, err in failed_pdfs:
        print(f"  - {fname}: {err}")
else:
    print("  (none)")

print(f"\nPDFs with unmatched DISCHARGE OF MORTGAGE entries ({len(unmatched_discharge_log)}):")
if unmatched_discharge_log:
    for fname, pid, disc in unmatched_discharge_log:
        print(
            f"  - {fname} (PID {pid}): discharge regnum {disc['regnum']} dated "
            f"{disc['date']} -- could not confidently match to a mortgage"
        )
else:
    print("  (none)")

print(f"\nProp IDs in database with NO matching PDF ({len(db_ids_without_pdf)}):")
if db_ids_without_pdf:
    for pid in db_ids_without_pdf:
        print(f"  - {pid}")
else:
    print("  (none)")

print(f"\nPDFs/PIDs with NO matching row in database ({len(pdf_ids_without_db_match)}):")
if pdf_ids_without_db_match:
    for pid in pdf_ids_without_db_match:
        print(f"  - {pid}")
else:
    print("  (none)")

print("\n" + "=" * 70)
print(f"Output written to: {OUTPUT_PATH}")
print("=" * 70)
